## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("Gold_fact_motorcycles")
logger.info("--- Starting Gold Layer: Creating fact_motorcycles ---")

## Creating fact view

In [0]:
try:
    logger.info("Step 1/4: Running SQL to create fact_motorcycles view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_motorcycles AS
    SELECT
    tw.car_num,
    m.manufacturer_key,
    vt.vehicle_type_key,
    f.fuel_type_key,
    o.ownership_key,
    tw.model_nm,
    tw.total_weight,
    tw.prod_year,
    tw.road_entry_dt,
    1 AS vehicle_count

    FROM transport_lakehouse.silver.silver_vehicles_two_wheeled tw
    LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
    ON CAST(tw.manufacturer_cd AS STRING) = CAST(m.manufacturer_cd AS STRING)
    LEFT JOIN transport_lakehouse.gold.dim_vehicle_type vt
    ON CAST(tw.vehicle_type_cd AS STRING) = CAST(vt.vehicle_type_cd AS STRING)
        AND tw.vehicle_type_nm = vt.vehicle_type_nm
    LEFT JOIN transport_lakehouse.gold.dim_fuel_type f
    ON tw.fuel_type_nm = f.fuel_type_nm
    LEFT JOIN transport_lakehouse.gold.dim_ownership o
    ON tw.ownership = o.ownership
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/4: View created successfully.")

except Exception as e:
    logger.error(f"Failed to create Gold Dimension: {str(e)}")
    raise e


## Missing keys check

In [0]:

logger.info("Step 2/4: Starting missing keys check...")

df_check = spark.sql("""
SELECT
  SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer,
  SUM(CASE WHEN vehicle_type_key IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_type,
  SUM(CASE WHEN fuel_type_key IS NULL THEN 1 ELSE 0 END) AS missing_fuel_type,
  SUM(CASE WHEN ownership_key IS NULL THEN 1 ELSE 0 END) AS missing_ownership
FROM transport_lakehouse.gold.fact_motorcycles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['missing_manufacturer']}, "
    f"vehicle_types: {results['missing_vehicle_type']}, "
    f"Fuel: {results['missing_fuel_type']}, "
    f"Ownership: {results['missing_ownership']}"
)

if (results['missing_manufacturer'] + results['missing_vehicle_type'] + 
    results['missing_fuel_type'] + results['missing_ownership']) > 0:
    logger.warning(f"Step 2/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 2/4: Finished missing keys check successfully. All keys are mapped. {missing_report}")
    

## Duplication check

In [0]:
logger.info("Step 3/4: Starting Duplication check...")

df_check = spark.sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_private_vehicles
""")

results = df_check.collect()[0]

if results['rows'] == results['distinct_cars']:
    logger.info(f"Step 2/4: Finished duplication check successfully. All rows are unique.")
else:
    logger.warning(f"Step 2/4: Finished with DUPLICATE ROWS!")


## Key null ratio check

In [0]:
logger.info("Step 4/4: Starting Key null ratio check...")

df_check = spark.sql("""
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN ownership_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS ownership_key_fill_rate,
  CAST(AVG(CASE WHEN fuel_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS fuel_type_key_fill_rate,
  CAST(AVG(CASE WHEN vehicle_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS vehicle_type_key_fill_rate
FROM transport_lakehouse.gold.fact_motorcycles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['manufacturer_key_fill_rate']}, "
    f"vehicle types: {results['vehicle_type_key_fill_rate']}, "
    f"Fuel: {results['fuel_type_key_fill_rate']}, "
    f"Ownership: {results['ownership_key_fill_rate']}"
)

if (results['manufacturer_key_fill_rate'] < 1 or 
    results['vehicle_type_key_fill_rate'] < 1 or 
    results['fuel_type_key_fill_rate'] < 1 or 
    results['ownership_key_fill_rate'] < 1 ):
    logger.warning(f"Step 4/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 4/4: Finished  Key null ratio successfully. All keys are mapped. {missing_report}")
    logger.info("--- Gold fact_private_vehicles Process Completed ---")
